# HDB Resale Data Pipeline - Part 1

## Source Extraction

This pipeline processes HDB resale transactions for January 2012 through December 2016. Original CSV files are retained unchanged; accepted and excluded records are written to separate outputs.

Run the cells in order from the repository root or the `part-1` folder using the environment described in the README. Findings below refer to the current source snapshot.

In [85]:
from pathlib import Path
import csv
import json
import shutil
import time
import pandas as pd
import numpy as np
import hashlib
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

### Source Configuration

Collection 189 comes from the assignment's source URL. Its metadata supplies the dataset IDs, so extraction does not depend on a manually maintained list.

Metadata is cached by default. `REFRESH_COLLECTION` refreshes that list; it does not replace existing CSV files.

In [52]:
COLLECTION_ID = 189

METADATA_API = "https://api-production.data.gov.sg/v2/public/api"
DOWNLOAD_API = "https://api-open.data.gov.sg/v1/public/api"

# Refresh the dataset list independently of the local CSV cache.
REFRESH_COLLECTION = False

### Storage Paths

Original files live in `data/raw/`; collection metadata lives in `data/metadata/`. Paths are resolved from the Part 1 directory so the notebook works from either supported working directory.

Existing directories are reused on reruns.

In [53]:
working_dir = Path.cwd()

if (working_dir / "hdb_resale_pipeline.ipynb").is_file():
    PART1_DIR = working_dir
elif (working_dir / "part-1" / "hdb_resale_pipeline.ipynb").is_file():
    PART1_DIR = working_dir / "part-1"
else:
    raise RuntimeError("Run from the project root or the part-1 folder.")

RAW_DIR = PART1_DIR / "data" / "raw"
METADATA_DIR = PART1_DIR / "data" / "metadata"

RAW_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_CACHE = METADATA_DIR / f"collection_{COLLECTION_ID}.json"

print(f"Raw files: {RAW_DIR}")
print(f"Metadata cache: {COLLECTION_CACHE}")

Raw files: c:\Users\mirza\Desktop\Personal Projects\hdb-data-engineering-technical-test\part-1\data\raw
Metadata cache: c:\Users\mirza\Desktop\Personal Projects\hdb-data-engineering-technical-test\part-1\data\metadata\collection_189.json


### Request Policy

Requests use a 60-second socket timeout and at most five attempts. Rate limits and selected server or connection failures are retried; other HTTP errors stop processing.

Download API calls are spaced 13 seconds apart. A numeric `Retry-After` value takes precedence over the fallback delay. Retries cover opening the response; a failure while streaming a file requires rerunning extraction.

In [54]:
_last_download_api_call = 0.0


def open_url(url):
    global _last_download_api_call

    for attempt in range(5):
        # Throttle preparation and polling calls, not the file transfer URL.
        if url.startswith(DOWNLOAD_API):
            elapsed = time.monotonic() - _last_download_api_call
            wait_seconds = max(0, 13 - elapsed)
            time.sleep(wait_seconds)
            _last_download_api_call = time.monotonic()

        try:
            request = Request(
                url,
                headers={"User-Agent": "hdb-resale-etl/0.1"},
            )
            return urlopen(request, timeout=60)

        except HTTPError as error:
            # Configuration and access errors should fail without repeated requests.
            if error.code not in {429, 500, 502, 503, 504}:
                raise RuntimeError(
                    f"Request failed with HTTP {error.code}."
                ) from None

            retry_after = error.headers.get("Retry-After", "")
            delay = (
                float(retry_after)
                if retry_after.isdigit()
                else 15 * (attempt + 1)
            )

        except (URLError, TimeoutError, OSError):
            delay = 15 * (attempt + 1)

        if attempt == 4:
            raise RuntimeError("Request failed after five attempts.")

        print(f"Request failed; retrying in {delay:.0f}s...", flush=True)
        time.sleep(delay)

### API Response Checks

The JSON helper checks the API response code before returning the payload. An HTTP success can still contain an application error, which stops processing with the service's message.

In [55]:
def api_json(url):
    with open_url(url) as response:
        payload = json.load(response)

    if payload.get("code", 0) != 0:
        raise RuntimeError(
            f"API error: {payload.get('errorMsg', 'Unknown error')}"
        )

    return payload

### Dataset Discovery

Discovery reads cached collection metadata unless a refresh is requested. Dataset IDs are checked before use in URLs and filenames. A newly fetched cache replaces the previous file only after writing completes.

In [56]:
def get_dataset_ids():
    if COLLECTION_CACHE.is_file() and not REFRESH_COLLECTION:
        payload = json.loads(
            COLLECTION_CACHE.read_text(encoding="utf-8")
        )
        print("Using cached collection metadata.")
    else:
        payload = api_json(
            f"{METADATA_API}/collections/{COLLECTION_ID}/metadata"
        )
        print("Retrieved collection metadata from the API.")

    dataset_ids = payload["data"]["collectionMetadata"]["childDatasets"]

    if not dataset_ids:
        raise RuntimeError("The collection contains no datasets.")

    for dataset_id in dataset_ids:
        valid_id = (
            isinstance(dataset_id, str)
            and dataset_id.startswith("d_")
            and len(dataset_id) == 34
            and all(
                character in "0123456789abcdef"
                for character in dataset_id[2:]
            )
        )

        if not valid_id:
            raise ValueError(f"Unexpected dataset ID: {dataset_id!r}")

    # Publish only the fully written metadata cache.
    if not COLLECTION_CACHE.is_file() or REFRESH_COLLECTION:
        temporary = COLLECTION_CACHE.with_suffix(".json.part")
        temporary.write_text(
            json.dumps(payload, indent=2),
            encoding="utf-8",
        )
        temporary.replace(COLLECTION_CACHE)

    return dataset_ids

### Discovered Sources

The dataset IDs below identify the files processed by extraction. The observed collection contains five sources; the count is reported rather than enforced so publisher changes remain visible.

In [57]:
dataset_ids = get_dataset_ids()

print(f"Found {len(dataset_ids)} datasets in collection {COLLECTION_ID}.")

for dataset_id in dataset_ids:
    print(dataset_id)

Using cached collection metadata.
Found 5 datasets in collection 189.
d_8b84c4ee58e3cfc0ece0d773c8ca6abc
d_43f493c6c50d54243cc1eab0df142d6a
d_2d5ff9ea31397b66239f245f57751537
d_ebc5ab87086db484f88045b47411ebc5
d_ea9ed51da2787afaf8e51f827c304208


### Source Download Function

Existing non-empty CSVs are reused, assuming local files are intact. Missing or empty files are requested without row or column filters.

Each transfer is streamed to a `.part` file. The downloaded size is compared with `Content-Length` when available, and the CSV header is checked before renaming. Failed transfers are removed. Download preparation is limited to ten polls.

In [58]:
def download_dataset(dataset_id):
    destination = RAW_DIR / f"{dataset_id}.csv"

    if destination.is_file() and destination.stat().st_size > 0:
        print(f"SKIPPED: {destination.name}", flush=True)
        return {
            "dataset_id": dataset_id,
            "status": "skipped",
            "path": destination,
        }

    print(f"Preparing download: {dataset_id}", flush=True)

    api_json(
        f"{DOWNLOAD_API}/datasets/{dataset_id}/initiate-download"
    )

    for _ in range(10):
        payload = api_json(
            f"{DOWNLOAD_API}/datasets/{dataset_id}/poll-download"
        )
        download_url = payload.get("data", {}).get("url")

        if download_url:
            break
    else:
        raise TimeoutError(
            f"Download not ready for {dataset_id}; rerun this step."
        )

    # An interrupted transfer must never look like a completed source file.
    temporary = destination.with_suffix(".csv.part")

    try:
        with open_url(download_url) as response:
            expected_bytes = response.headers.get("Content-Length")

            with temporary.open("wb") as output:
                shutil.copyfileobj(response, output)

        actual_bytes = temporary.stat().st_size

        if actual_bytes == 0:
            raise RuntimeError(f"Empty download for {dataset_id}.")

        if (
            expected_bytes is not None
            and actual_bytes != int(expected_bytes)
        ):
            raise RuntimeError(f"Incomplete download for {dataset_id}.")

        with temporary.open(
            encoding="utf-8-sig",
            newline="",
        ) as source:
            header = next(csv.reader(source), [])

        if "month" not in header or "resale_price" not in header:
            raise RuntimeError(
                f"Unexpected CSV header for {dataset_id}: {header}"
            )

        temporary.replace(destination)

    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(
        f"DOWNLOADED: {destination.name} ({actual_bytes:,} bytes)",
        flush=True,
    )

    return {
        "dataset_id": dataset_id,
        "status": "downloaded",
        "path": destination,
    }

### Run Source Extraction

Sources are processed sequentially. Completed files are skipped on reruns, allowing an interrupted extraction to resume from missing files. The summary records whether each source was downloaded or reused.

In [59]:
download_results = []

for dataset_id in dataset_ids:
    result = download_dataset(dataset_id)
    download_results.append(result)

downloaded = sum(
    result["status"] == "downloaded"
    for result in download_results
)

skipped = sum(
    result["status"] == "skipped"
    for result in download_results
)

print(f"\nComplete: {downloaded} downloaded, {skipped} skipped.")
print(f"Source files: {RAW_DIR}")

SKIPPED: d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv
SKIPPED: d_43f493c6c50d54243cc1eab0df142d6a.csv
SKIPPED: d_2d5ff9ea31397b66239f245f57751537.csv
SKIPPED: d_ebc5ab87086db484f88045b47411ebc5.csv
SKIPPED: d_ea9ed51da2787afaf8e51f827c304208.csv

Complete: 0 downloaded, 5 skipped.
Source files: c:\Users\mirza\Desktop\Personal Projects\hdb-data-engineering-technical-test\part-1\data\raw


### Source Schemas

Header inspection establishes the actual source columns before assigning types. Schema differences are retained during consolidation.

In [60]:
for dataset_id in dataset_ids:
    source_path = RAW_DIR / f"{dataset_id}.csv"

    with source_path.open(encoding="utf-8-sig", newline="") as source:
        headers = next(csv.reader(source))

    print(f"{source_path.name}:")
    print(headers)
    print()

d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']

d_43f493c6c50d54243cc1eab0df142d6a.csv:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

d_2d5ff9ea31397b66239f245f57751537.csv:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

d_ebc5ab87086db484f88045b47411ebc5.csv:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

d_ea9ed51da2787afaf8e51f827c304208.csv:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']



### Load Source Records

The sources share ten columns; two also provide `remaining_lease`. Consolidation retains all eleven attributes.

Categorical fields, block identifiers, and source remaining-lease values are read as text. Prices, floor areas, and commencement years use provisional numeric inference. Empty fields become nulls; other text is not automatically interpreted as missing.

In [61]:
import pandas as pd

text_columns = {
    "month": "string",
    "town": "string",
    "flat_type": "string",
    "block": "string",
    "street_name": "string",
    "storey_range": "string",
    "flat_model": "string",
    "remaining_lease": "string",
}

source_frames = []
source_summary = []

for dataset_id in dataset_ids:
    source_path = RAW_DIR / f"{dataset_id}.csv"

    headers = pd.read_csv(source_path, nrows=0).columns

    # Older schemas omit remaining_lease; apply types only to present columns.
    source_types = {
        column: dtype
        for column, dtype in text_columns.items()
        if column in headers
    }

    frame = pd.read_csv(
        source_path,
        dtype=source_types,
        keep_default_na=False,
        na_values=[""],
    )

    source_frames.append(frame)

    source_summary.append({
        "file": source_path.name,
        "rows": len(frame),
        "first_month": frame["month"].min(),
        "last_month": frame["month"].max(),
        "columns": len(frame.columns),
    })

display(pd.DataFrame(source_summary))

,file,rows,first_month,last_month,columns
0,d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv,240345,2017-01,2026-09,11
1,d_43f493c6c50d54243cc1eab0df142d6a.csv,369651,2000-01,2012-02,10
2,d_2d5ff9ea31397b66239f245f57751537.csv,52203,2012-03,2014-12,10
3,d_ebc5ab87086db484f88045b47411ebc5.csv,287196,1990-01,1999-12,10
4,d_ea9ed51da2787afaf8e51f827c304208.csv,37153,2015-01,2016-12,11


## Data Quality Requirements

<h3 style="color: #4EA1FF;">1. Combine Sources and Retain All Attributes</h3>

Concatenation retains the union of source columns. Valid months within January 2012–December 2016 enter `master`; missing or malformed months enter date quarantine. Valid months outside the period are scope exclusions.

The combined row index remains the record reference for this run. The three routing counts must reconcile to the combined source count.

In [62]:
combined = pd.concat(
    source_frames,
    ignore_index=True,
    sort=False,
)

valid_month_format = combined["month"].str.fullmatch(
    r"\d{4}-(0[1-9]|1[0-2])",
    na=False,
)

within_required_period = (
    combined["month"]
    .between("2012-01", "2016-12")
    .fillna(False)
)

in_scope = valid_month_format & within_required_period
invalid_date = ~valid_month_format
outside_period = valid_month_format & ~within_required_period

# Preserve combined-source indices for reconciliation within this run.
master = combined.loc[in_scope].copy()

quarantined_dates = combined.loc[invalid_date].copy()
quarantined_dates.insert(
    0,
    "source_record_id",
    quarantined_dates.index,
)

quarantined_dates["quarantine_reasons"] = "invalid_month_format"

missing_month = (
    quarantined_dates["month"].isna()
    | quarantined_dates["month"]
    .str.strip()
    .eq("")
    .fillna(False)
)

quarantined_dates.loc[
    missing_month,
    "quarantine_reasons",
] = "missing_month"

scope_exclusion_count = int(outside_period.sum())

expected_columns = {
    column
    for frame in source_frames
    for column in frame.columns
}

if set(master.columns) != expected_columns:
    raise ValueError("Source columns were lost during consolidation.")

if master.empty:
    raise ValueError("No records found within the required date range.")

accounted_rows = (
    len(master)
    + len(quarantined_dates)
    + scope_exclusion_count
)

if accounted_rows != len(combined):
    raise ValueError("Source routing counts do not reconcile.")

print(f"Combined source rows: {len(combined):,}")
print(f"Outside-period exclusions: {scope_exclusion_count:,}")
print(f"Quarantined for invalid dates: {len(quarantined_dates):,}")
print(f"Master dataset rows: {len(master):,}")
print(
    f"Month coverage: "
    f"{master['month'].min()} to {master['month'].max()}"
)
print(f"Columns retained: {len(master.columns)}")

display(master.head())
display(quarantined_dates.head())

Combined source rows: 986,548
Outside-period exclusions: 894,004
Quarantined for invalid dates: 0
Master dataset rows: 92,544
Month coverage: 2012-01 to 2016-12
Columns retained: 11


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
606808,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,<NA>,257800.0
606809,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,<NA>,263000.0
606810,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,<NA>,275000.0
606811,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,<NA>,260000.0
606812,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,<NA>,226000.0


,source_record_id,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,quarantine_reasons


<h3 style="color: #4EA1FF;">2. Profile the Master Dataset</h3>

Profiling describes the consolidated inputs before cleaning. `master` remains unchanged.

#### Baseline

Row count, column count, month coverage, inferred types, and memory usage establish the starting point for subsequent reconciliation. Inferred types describe how values were loaded; they do not establish validity.

In [63]:
baseline = pd.DataFrame([{
    "rows": len(master),
    "columns": len(master.columns),
    "first_month": master["month"].min(),
    "last_month": master["month"].max(),
    "memory_mb": round(
        master.memory_usage(index=True, deep=True).sum() / (1024 ** 2),
        2,
    ),
}])

column_types = (
    master.dtypes
    .astype(str)
    .rename_axis("column")
    .reset_index(name="dtype")
)

display(baseline)
display(column_types)

,rows,columns,first_month,last_month,memory_mb
0,92544,11,2012-01,2016-12,42.91


,column,dtype
0,month,string
1,town,string
2,flat_type,string
3,block,string
4,street_name,string
5,storey_range,string
6,floor_area_sqm,float64
7,flat_model,string
8,lease_commence_date,int64
9,remaining_lease,string


### Completeness

Nulls and whitespace-only text are counted separately. Missing remaining lease is also grouped by year to distinguish an absent source attribute from missing values within a supplied column.

In [64]:
null_counts = master.isna().sum()

blank_counts = pd.Series(
    0,
    index=master.columns,
    dtype="int64",
)

for column in master.select_dtypes(include=["string", "object"]).columns:
    blank_counts[column] = (
        master[column]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
        .sum()
    )

completeness = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": (null_counts / len(master) * 100).round(2),
    "blank_count": blank_counts,
    "blank_pct": (blank_counts / len(master) * 100).round(2),
})

completeness.index.name = "column"

display(
    completeness.sort_values(
        ["null_count", "blank_count"],
        ascending=False,
    )
)

# Check whether missing lease values follow the source schema boundaries.
lease_by_year = (
    master.assign(
        year=master["month"].str[:4],
        lease_missing=master["remaining_lease"].isna(),
    )
    .groupby("year")
    .agg(
        rows=("lease_missing", "size"),
        remaining_lease_nulls=("lease_missing", "sum"),
    )
)

lease_by_year["null_pct"] = (
    lease_by_year["remaining_lease_nulls"]
    / lease_by_year["rows"]
    * 100
).round(2)

display(lease_by_year)

,null_count,null_pct,blank_count,blank_pct
column,,,,
remaining_lease,55391,59.85,0,0.0
month,0,0.00,0,0.0
town,0,0.00,0,0.0
flat_type,0,0.00,0,0.0
block,0,0.00,0,0.0
street_name,0,0.00,0,0.0
storey_range,0,0.00,0,0.0
floor_area_sqm,0,0.00,0,0.0
flat_model,0,0.00,0,0.0


,rows,remaining_lease_nulls,null_pct
year,,,
2012,23198,23198,100.0
2013,16097,16097,100.0
2014,16096,16096,100.0
2015,17780,0,0.0
2016,19373,0,0.0


### Completeness Findings

`remaining_lease` is missing in 55,391 records (59.85%), covering all 2012–2014 transactions. The sources for those years omit the attribute. All 2015–2016 records supply a value.

These structural nulls are retained and a lease balance is calculated separately. The other ten columns contain no nulls or whitespace-only values.

### Monthly Coverage

Observed counts are reindexed against the 60 expected months so an empty month appears explicitly. Month-to-month volume changes are reported for investigation, without using volume alone as an exclusion rule.

In [65]:
expected_months = pd.period_range(
    start="2012-01",
    end="2016-12",
    freq="M",
).astype(str)

# Include absent months explicitly rather than showing only observed months.
monthly_profile = (
    master.groupby("month")
    .size()
    .reindex(expected_months, fill_value=0)
    .rename("rows")
    .rename_axis("month")
    .to_frame()
)

monthly_profile["change_pct"] = (
    monthly_profile["rows"]
    .pct_change(fill_method=None)
    .mul(100)
    .round(2)
)

missing_months = monthly_profile.index[
    monthly_profile["rows"].eq(0)
].tolist()

print(f"Expected months: {len(expected_months)}")
print(f"Months with records: {monthly_profile['rows'].gt(0).sum()}")
print(f"Missing months: {missing_months}")

display(monthly_profile)

Expected months: 60
Months with records: 60
Missing months: []


,rows,change_pct
month,,
2012-01,1559,NaN
2012-02,1629,4.49
2012-03,2360,44.87
2012-04,2155,-8.69
2012-05,2323,7.80
2012-06,1993,-14.21
2012-07,2179,9.33
2012-08,2075,-4.77
2012-09,1760,-15.18


### Coverage Findings

All 60 required months contain records. Monthly counts range from 886 to 2,360. The largest decrease is 45.74% in February 2013; the largest increase is 50.57% in March 2014.

There are no empty months. This check does not establish that every transaction is present.

### Categorical Distributions

Distinct values and frequencies are profiled for town, flat type, flat model, and storey range. January 2012 supplies the reference sets required by the assignment. Values outside those sets are counted before exclusion.

In [66]:
category_columns = [
    "town",
    "flat_type",
    "flat_model",
    "storey_range",
]

january_reference = master.loc[master["month"].eq("2012-01")]

if january_reference.empty:
    raise ValueError("January 2012 reference records are missing.")

# Freeze reference values from January 2012 before any cleaning.
reference_sets = {}
category_summary = []
outside_reference_tables = []

for column in category_columns:
    reference_sets[column] = set(
        january_reference[column].dropna().unique()
    )

    counts = (
        master[column]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="rows")
    )

    counts["in_january_reference"] = counts["value"].isin(
        reference_sets[column]
    )

    outside = counts.loc[
        ~counts["in_january_reference"]
    ].copy()

    outside.insert(0, "column", column)
    outside_reference_tables.append(outside)

    category_summary.append({
        "column": column,
        "distinct_values": master[column].nunique(dropna=True),
        "january_reference_values": len(reference_sets[column]),
        "rows_outside_reference": int(outside["rows"].sum()),
    })

    print(f"\n{column}")
    display(counts)

display(pd.DataFrame(category_summary))

outside_reference = pd.concat(
    outside_reference_tables,
    ignore_index=True,
)

print("\nValues outside the January 2012 reference:")
display(outside_reference)


town


,value,rows,in_january_reference
0,JURONG WEST,7573,True
1,WOODLANDS,7399,True
2,TAMPINES,6728,True
3,BEDOK,6071,True
4,YISHUN,5937,True
5,SENGKANG,5896,True
6,HOUGANG,4735,True
7,ANG MO KIO,4558,True
8,CHOA CHU KANG,3928,True
9,BUKIT BATOK,3773,True



flat_type


,value,rows,in_january_reference
0,4 ROOM,36535,True
1,3 ROOM,26307,True
2,5 ROOM,21368,True
3,EXECUTIVE,7295,True
4,2 ROOM,956,True
5,1 ROOM,56,True
6,MULTI-GENERATION,27,True



flat_model


...


storey_range


,value,rows,in_january_reference
0,04 TO 06,21214,True
1,07 TO 09,18765,True
2,01 TO 03,17466,True
3,10 TO 12,16028,True
4,13 TO 15,6704,True
5,16 TO 18,2706,True
6,01 TO 05,2700,False
7,06 TO 10,2474,False
8,11 TO 15,1259,False
9,19 TO 21,1156,True


,column,distinct_values,january_reference_values,rows_outside_reference
0,town,26,26,0
1,flat_type,7,7,0
2,flat_model,20,13,492
3,storey_range,25,12,6975



Values outside the January 2012 reference:


,column,value,rows,in_january_reference
0,flat_model,DBSS,277,False
1,flat_model,Type S1,138,False
2,flat_model,Type S2,55,False
3,flat_model,Improved-Maisonette,10,False
4,flat_model,Premium Maisonette,6,False
5,flat_model,Premium Apartment Loft,5,False
6,flat_model,2-room,1,False
7,storey_range,01 TO 05,2700,False
8,storey_range,06 TO 10,2474,False
9,storey_range,11 TO 15,1259,False


### Category Findings

All 26 towns and seven flat types match January 2012. Seven additional flat models affect 492 records; thirteen additional storey ranges affect 6,975. These per-field counts can overlap.

Reference membership follows the assignment's policy. A category absent from one month's transactions may still be legitimate. Five-storey bands are not mapped to three-storey bands because the exact floor is unknown.

### Numeric Distributions

Price, floor area, and commencement year are checked for missing or nonnumeric values and profiled using ranges and percentiles. Basic checks count nonpositive values, fractional commencement years, and commencement after the transaction year.

Price and area extremes require comparison with similar transactions before exclusion.

In [67]:
numeric_columns = [
    "resale_price",
    "floor_area_sqm",
    "lease_commence_date",
]

# Coercion exposes invalid numeric text without changing master.
numeric_values = master[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

numeric_summary = numeric_values.describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99],
).T

numeric_summary["source_nulls"] = master[numeric_columns].isna().sum()

numeric_summary["nonnumeric_count"] = (
    master[numeric_columns].notna()
    & numeric_values.isna()
).sum()

display(numeric_summary)

transaction_year = master["month"].str[:4].astype("int64")
lease_year = numeric_values["lease_commence_date"]

numeric_checks = pd.DataFrame([
    {
        "check": "Nonpositive resale price",
        "affected_rows": numeric_values["resale_price"].le(0).sum(),
    },
    {
        "check": "Nonpositive floor area",
        "affected_rows": numeric_values["floor_area_sqm"].le(0).sum(),
    },
    {
        "check": "Nonpositive lease commencement year",
        "affected_rows": lease_year.le(0).sum(),
    },
    {
        "check": "Non-integer lease commencement year",
        "affected_rows": (
            lease_year.notna() & lease_year.mod(1).ne(0)
        ).sum(),
    },
    {
        "check": "Lease commencement after transaction year",
        "affected_rows": lease_year.gt(transaction_year).sum(),
    },
])

display(numeric_checks)

,count,mean,std,min,1%,25%,50%,75%,99%,max,source_nulls,nonnumeric_count
resale_price,92544.0,450938.971730,128181.301162,190000.0,260000.0,357000.0,428000.0,515000.0,848000.0,1150000.0,0,0
floor_area_sqm,92544.0,96.569115,24.682292,31.0,50.0,74.0,95.0,111.0,151.0,280.0,0,0
lease_commence_date,92544.0,1990.072701,10.446719,1966.0,1969.0,1983.0,1988.0,1999.0,2012.0,2013.0,0,0


,check,affected_rows
0,Nonpositive resale price,0
1,Nonpositive floor area,0
2,Nonpositive lease commencement year,0
3,Non-integer lease commencement year,0
4,Lease commencement after transaction year,0


### Numeric Findings

All 92,544 records have numeric, populated prices, floor areas, and commencement years. The basic checks found no failures.

Prices range from $190,000 to $1,150,000, with a $428,000 median. Floor areas range from 31 to 280 m², and commencement years from 1966 to 2013. These ranges inform later screening; they are not rejection thresholds.

### Duplicate Profile

The composite key is fixed from the original master columns, excluding `resale_price`. Exact duplicate copies are counted separately from repeated keys with different prices.

Surplus counts represent records excluded when retaining one highest-priced record per key. Null key attributes remain included in the grouped analysis.

In [68]:
# Derived columns added later must not extend the assignment key.
key_columns = [
    column
    for column in master.columns
    if column != "resale_price"
]

exact_duplicate_mask = master.duplicated(keep=False)
key_duplicate_mask = master.duplicated(
    subset=key_columns,
    keep=False,
)

duplicate_groups = (
    master.loc[key_duplicate_mask]
    .groupby(
        key_columns,
        # Include groups whose source remaining_lease is null.
        dropna=False,
        observed=True,
    )
    .agg(
        records=("resale_price", "size"),
        distinct_prices=("resale_price", "nunique"),
        lowest_price=("resale_price", "min"),
        highest_price=("resale_price", "max"),
    )
    .reset_index()
)

duplicate_summary = pd.DataFrame([{
    "rows_in_exact_duplicate_groups": int(exact_duplicate_mask.sum()),
    "surplus_exact_duplicates": int(master.duplicated().sum()),
    "rows_in_repeated_key_groups": int(key_duplicate_mask.sum()),
    "repeated_key_groups": len(duplicate_groups),
    "groups_with_conflicting_prices": int(
        duplicate_groups["distinct_prices"].gt(1).sum()
    ),
    "surplus_records_by_key": int(
        master.duplicated(subset=key_columns).sum()
    ),
}])

display(duplicate_summary)

display(
    duplicate_groups.sort_values(
        ["distinct_prices", "records"],
        ascending=False,
    ).head(20)
)

,rows_in_exact_duplicate_groups,surplus_exact_duplicates,rows_in_repeated_key_groups,repeated_key_groups,groups_with_conflicting_prices,surplus_records_by_key
0,546,273,3154,1559,1294,1595


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,records,distinct_prices,lowest_price,highest_price
5,2012-01,BUKIT MERAH,3 ROOM,37,JLN RUMAH TINGGI,04 TO 06,53.0,Standard,1969,<NA>,3,3,290000.0,318000.0
36,2012-02,WOODLANDS,3 ROOM,12,MARSILING LANE,07 TO 09,65.0,Improved,1976,<NA>,3,3,288000.0,310000.0
45,2012-03,BEDOK,4 ROOM,80,BEDOK NTH RD,01 TO 05,91.0,New Generation,1978,<NA>,3,3,400000.0,422800.0
64,2012-03,JURONG EAST,3 ROOM,101,JURONG EAST ST 13,11 TO 15,68.0,New Generation,1983,<NA>,3,3,332000.0,346000.0
68,2012-03,JURONG EAST,3 ROOM,302,JURONG EAST ST 32,01 TO 05,68.0,New Generation,1983,<NA>,3,3,310000.0,331000.0
69,2012-03,JURONG WEST,3 ROOM,209,BOON LAY PL,11 TO 15,65.0,Improved,1976,<NA>,3,3,303000.0,310500.0
127,2012-04,CLEMENTI,3 ROOM,309,CLEMENTI AVE 4,06 TO 10,67.0,New Generation,1980,<NA>,3,3,366000.0,400000.0
133,2012-04,JURONG WEST,5 ROOM,625,JURONG WEST ST 61,06 TO 10,110.0,Improved,2001,<NA>,3,3,490000.0,505000.0
207,2012-05,SEMBAWANG,4 ROOM,467,ADMIRALTY DR,11 TO 15,102.0,Premium Apartment,2001,<NA>,3,3,419000.0,440000.0
227,2012-05,YISHUN,3 ROOM,734,YISHUN AVE 5,06 TO 10,67.0,New Generation,1985,<NA>,3,3,336000.0,360000.0


### Duplicate Findings

There are 546 records in exact duplicate groups, including 273 surplus copies. The specified composite key identifies 3,154 records across 1,559 repeated-key groups; 1,294 groups contain conflicting prices. Retaining one record per key would exclude 1,595 records before other validation.

For the January 2012 Bukit Merah example, the rule retains $318,000 from three records priced between $290,000 and $318,000. Equal maxima retain the first source record. These coarse attributes may describe distinct transactions; deduplication follows the assignment's assumption.

### Preliminary Price Screen

Price per square metre is compared within year, town, and flat type. The initial rule uses fences at `Q1 - 3 × IQR` and `Q3 + 3 × IQR`, with at least 30 records and a positive IQR per group.

This is exploratory screening. Thresholds are recalculated after validation and deduplication before candidates are quarantined. Location, model, floor level, and lease differences are not fully captured by these groups.

In [69]:
# Preliminary thresholds describe master; final screening refits after exclusions.
price_profile = master.assign(
    year=master["month"].str[:4],
    price_per_sqm=(
        numeric_values["resale_price"]
        / numeric_values["floor_area_sqm"]
    ),
)

group_columns = ["year", "town", "flat_type"]

price_group_stats = (
    price_profile.groupby(group_columns, observed=True)
    .agg(
        group_rows=("price_per_sqm", "count"),
        q1=("price_per_sqm", lambda values: values.quantile(0.25)),
        median=("price_per_sqm", "median"),
        q3=("price_per_sqm", lambda values: values.quantile(0.75)),
    )
    .reset_index()
)

price_group_stats["iqr"] = (
    price_group_stats["q3"] - price_group_stats["q1"]
)

price_group_stats["lower_fence"] = (
    price_group_stats["q1"] - 3 * price_group_stats["iqr"]
)

price_group_stats["upper_fence"] = (
    price_group_stats["q3"] + 3 * price_group_stats["iqr"]
)

price_profile = price_profile.merge(
    price_group_stats,
    on=group_columns,
    how="left",
    validate="many_to_one",
)

price_profile["screening_eligible"] = (
    price_profile["group_rows"].ge(30)
    & price_profile["iqr"].gt(0)
    & price_profile["price_per_sqm"].notna()
)

price_profile["potential_price_anomaly"] = (
    price_profile["screening_eligible"]
    & (
        price_profile["price_per_sqm"].lt(price_profile["lower_fence"])
        | price_profile["price_per_sqm"].gt(price_profile["upper_fence"])
    )
)

print(
    "Records eligible for screening:",
    int(price_profile["screening_eligible"].sum()),
)
print(
    "Records not evaluated:",
    int((~price_profile["screening_eligible"]).sum()),
)
print(
    "Potential price anomalies:",
    int(price_profile["potential_price_anomaly"].sum()),
)

display(
    price_profile.loc[
        price_profile["potential_price_anomaly"],
        [
            "month",
            "town",
            "flat_type",
            "floor_area_sqm",
            "resale_price",
            "price_per_sqm",
            "group_rows",
            "lower_fence",
            "upper_fence",
        ],
    ].head(20)
)

Records eligible for screening: 90775
Records not evaluated: 1769
Potential price anomalies: 414


,month,town,flat_type,floor_area_sqm,resale_price,price_per_sqm,group_rows,lower_fence,upper_fence
839,2012-01,KALLANG/WHAMPOA,3 ROOM,79.0,705000.0,8924.050633,354,3150.933380,7811.158939
1606,2012-02,ANG MO KIO,4 ROOM,90.0,642000.0,7133.333333,299,3235.058636,6966.503016
1676,2012-02,BEDOK,4 ROOM,85.0,598000.0,7035.294118,457,3134.852735,6433.099579
2531,2012-02,QUEENSTOWN,2 ROOM,43.0,298000.0,6930.232558,49,4760.869565,6891.304348
3298,2012-03,ANG MO KIO,4 ROOM,91.0,635000.0,6978.021978,299,3235.058636,6966.503016
3583,2012-03,BUKIT BATOK,4 ROOM,90.0,602000.0,6688.888889,389,2440.476190,6607.142857
3635,2012-03,BUKIT MERAH,3 ROOM,60.0,525000.0,8750.000000,337,3028.248588,8617.702448
4008,2012-03,GEYLANG,3 ROOM,60.0,443888.0,7398.133333,348,3271.395314,7246.690577
4450,2012-03,KALLANG/WHAMPOA,3 ROOM,80.0,688000.0,8600.000000,354,3150.933380,7811.158939
4451,2012-03,KALLANG/WHAMPOA,3 ROOM,83.0,698000.0,8409.638554,354,3150.933380,7811.158939


### Preliminary Screen Findings

The initial screen evaluates 90,775 records and flags 414 candidates (0.46% of evaluated records). Another 1,769 records are unassessed because their groups are too small or have zero IQR.

The sample displays prices above the upper fence. These flags remain preliminary; later exclusions change the comparison population.

### Profiling Summary

The master contains 92,544 records, eleven attributes, and all 60 required months. Remaining-lease nulls align with source schema differences; other fields are complete.

The reference screen finds 492 flat-model failures and 6,975 storey-range failures. Basic numeric checks find none. Duplicate profiling identifies 1,595 surplus records, and preliminary price screening flags 414 candidates.

These are independent check counts and may overlap. Record routing, rather than the sum of profile counts, determines final dispositions.

<h3 style="color: #4EA1FF;">3. Validate Dates and January 2012 Reference Categories</h3>

#### Date and Category Rules

Dates must use `YYYY-MM` and fall within the assessment period. January 2012 establishes date representation, rather than literal allowed dates; restricting every transaction to January would contradict the required scope.

Town, flat type, flat model, and storey range use exact January 2012 membership. Each rule has a separate failure flag so records can retain multiple reasons. Date checks here also confirm the invariant established during source routing.

In [70]:
# Align rule flags with source references; True denotes a failure.
validation_checks = pd.DataFrame(index=master.index)

validation_checks["invalid_month_format"] = (
    ~master["month"].str.fullmatch(
        r"\d{4}-(0[1-9]|1[0-2])",
        na=False,
    )
)

validation_checks["month_outside_required_period"] = (
    ~master["month"]
    .between("2012-01", "2016-12")
    .fillna(False)
)

for column in category_columns:
    validation_checks[f"{column}_outside_january_reference"] = (
        ~master[column].isin(reference_sets[column])
    )

validation_summary = (
    validation_checks.sum()
    .rename_axis("rule")
    .reset_index(name="failed_rows")
)

validation_summary["failed_pct"] = (
    validation_summary["failed_rows"] / len(master) * 100
).round(2)

failed_any_rule = validation_checks.any(axis=1)

display(validation_summary)

print(f"Records failing at least one rule: {failed_any_rule.sum():,}")
print(f"Records passing all rules: {(~failed_any_rule).sum():,}")

,rule,failed_rows,failed_pct
0,invalid_month_format,0,0.0
1,month_outside_required_period,0,0.0
2,town_outside_january_reference,0,0.0
3,flat_type_outside_january_reference,0,0.0
4,flat_model_outside_january_reference,492,0.53
5,storey_range_outside_january_reference,6975,7.54


Records failing at least one rule: 7,411
Records passing all rules: 85,133


### Reference Validation Findings

Date, town, and flat-type checks pass throughout. Flat-model membership fails for 492 records and storey-range membership for 6,975, affecting 7,411 distinct records. Fifty-six fail both checks.

The remaining 85,133 records proceed to lease calculation, duplicate handling, and price screening. Reference failures identify policy exclusions, without proving the source categories are incorrect.

### Reference Failure Routing

Failed records retain their source values, run-specific record reference, and all applicable reference failure reasons. Passing records keep the original columns so diagnostics cannot change the duplicate key.

In [71]:
# Keep all failed reference rules, not just the first one.
failure_reasons = validation_checks.apply(
    lambda row: "; ".join(row.index[row]),
    axis=1,
)

quarantined_validation = master.loc[failed_any_rule].copy()

quarantined_validation.insert(
    0,
    "source_record_id",
    quarantined_validation.index,
)

quarantined_validation["quarantine_reasons"] = (
    failure_reasons.loc[failed_any_rule]
)

validated = master.loc[~failed_any_rule].copy()

if len(validated) + len(quarantined_validation) != len(master):
    raise ValueError("Validation outputs do not reconcile to the master.")

print(f"Passing reference validation: {len(validated):,}")
print(f"Quarantined for validation: {len(quarantined_validation):,}")

display(
    quarantined_validation[
        [
            "source_record_id",
            "month",
            "town",
            "flat_model",
            "storey_range",
            "quarantine_reasons",
        ]
    ].head(10)
)

Passing reference validation: 85,133
Quarantined for validation: 7,411


,source_record_id,month,town,flat_model,storey_range,quarantine_reasons
609754,609754,2012-02,TOA PAYOH,Improved,37 TO 39,storey_range_outside_january_reference
609996,609996,2012-03,ANG MO KIO,Improved,06 TO 10,storey_range_outside_january_reference
609997,609997,2012-03,ANG MO KIO,Improved,01 TO 05,storey_range_outside_january_reference
609998,609998,2012-03,ANG MO KIO,New Generation,06 TO 10,storey_range_outside_january_reference
609999,609999,2012-03,ANG MO KIO,New Generation,01 TO 05,storey_range_outside_january_reference
610000,610000,2012-03,ANG MO KIO,New Generation,06 TO 10,storey_range_outside_january_reference
610001,610001,2012-03,ANG MO KIO,New Generation,01 TO 05,storey_range_outside_january_reference
610002,610002,2012-03,ANG MO KIO,New Generation,01 TO 05,storey_range_outside_january_reference
610003,610003,2012-03,ANG MO KIO,New Generation,01 TO 05,storey_range_outside_january_reference
610004,610004,2012-03,ANG MO KIO,New Generation,11 TO 15,storey_range_outside_january_reference


### Validation Reconciliation

Date and category failures are combined into one quarantine dataset. Each source reference must appear once. Passing records, quarantine, and scope exclusions must reconcile to all combined source rows.

In [72]:
# Rebuild from stage outputs so reruns do not append the same failures.
quarantined = pd.concat(
    [quarantined_dates, quarantined_validation],
    ignore_index=True,
    sort=False,
)

if quarantined["source_record_id"].duplicated().any():
    raise ValueError("A source record appears more than once in quarantine.")

accounted_rows = (
    len(validated)
    + len(quarantined)
    + scope_exclusion_count
)

if accounted_rows != len(combined):
    raise ValueError("Validation outputs do not reconcile to source rows.")

print(f"Passing validation: {len(validated):,}")
print(f"Total validation quarantine: {len(quarantined):,}")
print(f"Outside-period exclusions: {scope_exclusion_count:,}")

Passing validation: 85,133
Total validation quarantine: 7,411
Outside-period exclusions: 894,004


<h3 style="color: #4EA1FF;">4. Calculate Remaining Lease</h3>

The calculation assumes a 99-year lease starting on 1 January of the supplied commencement year, measured at the start of the resale month. The source has no start month, so the month balance depends on this assumption.

Whole-month arithmetic produces completed years and a month remainder without rounding up. The source `remaining_lease` is preserved. Invalid commencement years or balances outside the term are quarantined.

In [73]:
lease_candidates = validated.copy()

transaction_year = (
    lease_candidates["month"].str[:4].astype("int64")
)

transaction_month = (
    lease_candidates["month"].str[5:7].astype("int64")
)

lease_start_year = pd.to_numeric(
    lease_candidates["lease_commence_date"],
    errors="coerce",
)

valid_start_year = (
    lease_start_year.notna()
    & lease_start_year.gt(0)
    & lease_start_year.mod(1).eq(0)
)

# January commencement and the start of the resale month define elapsed time.
elapsed_months = (
    (transaction_year - lease_start_year) * 12
    + (transaction_month - 1)
)

remaining_months = 99 * 12 - elapsed_months

lease_checks = pd.DataFrame({
    "invalid_lease_commencement_year": ~valid_start_year,
    "remaining_lease_outside_99_year_term": (
        valid_start_year
        & ~remaining_months.between(0, 99 * 12)
    ),
})

failed_lease = lease_checks.any(axis=1)

lease_failure_reasons = lease_checks.apply(
    lambda row: "; ".join(row.index[row]),
    axis=1,
)

quarantined_lease = lease_candidates.loc[failed_lease].copy()

quarantined_lease.insert(
    0,
    "source_record_id",
    quarantined_lease.index,
)

quarantined_lease["quarantine_reasons"] = (
    lease_failure_reasons.loc[failed_lease]
)

lease_calculated = lease_candidates.loc[~failed_lease].copy()

accepted_months = remaining_months.loc[~failed_lease]

# Split whole months into completed years and a 0–11 month remainder.
lease_calculated["remaining_lease_years"] = (
    accepted_months // 12
).astype("int64")

lease_calculated["remaining_lease_months"] = (
    accepted_months % 12
).astype("int64")

quarantined = pd.concat(
    [
        quarantined_dates,
        quarantined_validation,
        quarantined_lease,
    ],
    ignore_index=True,
    sort=False,
)

if quarantined["source_record_id"].duplicated().any():
    raise ValueError("A source record appears more than once in quarantine.")

accounted_rows = (
    len(lease_calculated)
    + len(quarantined)
    + scope_exclusion_count
)

if accounted_rows != len(combined):
    raise ValueError("Lease processing outputs do not reconcile.")

print(f"Remaining lease calculated: {len(lease_calculated):,}")
print(f"Quarantined for lease failures: {len(quarantined_lease):,}")

display(
    lease_calculated[
        [
            "month",
            "lease_commence_date",
            "remaining_lease",
            "remaining_lease_years",
            "remaining_lease_months",
        ]
    ].head(10)
)

Remaining lease calculated: 85,133
Quarantined for lease failures: 0


,month,lease_commence_date,remaining_lease,remaining_lease_years,remaining_lease_months
606808,2012-01,1979,<NA>,66,0
606809,2012-01,1978,<NA>,65,0
606810,2012-01,1978,<NA>,65,0
606811,2012-01,1986,<NA>,73,0
606812,2012-01,1986,<NA>,73,0
606813,2012-01,1980,<NA>,67,0
606814,2012-01,1986,<NA>,73,0
606815,2012-01,1976,<NA>,63,0
606816,2012-01,1981,<NA>,68,0
606817,2012-01,1979,<NA>,66,0


### Lease Findings

All 85,133 reference-passing records receive calculated lease balances. No commencement-year or term-bound failures are found. Calculated balances are available even where the source remaining lease is null.

<h3 style="color: #4EA1FF;">5. Retain the Highest Price per Composite Key</h3>

Duplicate handling uses the original source key fixed during profiling. Calculated lease attributes and diagnostics are excluded from that key.

A stable descending price sort retains the highest-priced record per key and the first source record when maximum prices tie. Surplus records are quarantined. Counts are calculated on the current input because reference exclusions have already changed the population.

In [74]:
# Stable ordering breaks equal-price ties using the original source order.
ranked = lease_calculated.sort_values(
    "resale_price",
    ascending=False,
    kind="stable",
)

duplicate_mask = ranked.duplicated(
    subset=key_columns,
    keep="first",
)

quarantined_duplicates = ranked.loc[duplicate_mask].copy()

quarantined_duplicates.insert(
    0,
    "source_record_id",
    quarantined_duplicates.index,
)

quarantined_duplicates["quarantine_reasons"] = (
    "duplicate_composite_key"
)

deduplicated = ranked.loc[~duplicate_mask].copy().sort_index()

quarantined = pd.concat(
    [
        quarantined_dates,
        quarantined_validation,
        quarantined_lease,
        quarantined_duplicates,
    ],
    ignore_index=True,
    sort=False,
)

if deduplicated.duplicated(subset=key_columns).any():
    raise ValueError("Repeated composite keys remain after deduplication.")

if quarantined["source_record_id"].duplicated().any():
    raise ValueError("A source record appears more than once in quarantine.")

if (
    len(deduplicated)
    + len(quarantined_duplicates)
    != len(lease_calculated)
):
    raise ValueError("Duplicate processing counts do not reconcile.")

accounted_rows = (
    len(deduplicated)
    + len(quarantined)
    + scope_exclusion_count
)

if accounted_rows != len(combined):
    raise ValueError("Pipeline outputs do not reconcile to source rows.")

print(f"Records before deduplication: {len(lease_calculated):,}")
print(f"Surplus records quarantined: {len(quarantined_duplicates):,}")
print(f"Records retained: {len(deduplicated):,}")
print(f"Total quarantined so far: {len(quarantined):,}")

display(
    quarantined_duplicates[
        [
            "source_record_id",
            "month",
            "town",
            "flat_type",
            "block",
            "resale_price",
            "quarantine_reasons",
        ]
    ].head(10)
)

Records before deduplication: 85,133
Surplus records quarantined: 1,383
Records retained: 83,750
Total quarantined so far: 8,794


,source_record_id,month,town,flat_type,block,resale_price,quarantine_reasons
638790,638790,2013-07,BUKIT TIMAH,EXECUTIVE,7,950000.0,duplicate_composite_key
639696,639696,2013-07,TOA PAYOH,5 ROOM,81,880000.0,duplicate_composite_key
638630,638630,2013-07,BISHAN,EXECUTIVE,187,878000.0,duplicate_composite_key
630313,630313,2013-01,BUKIT MERAH,4 ROOM,76A,828000.0,duplicate_composite_key
620464,620464,2012-07,SERANGOON,EXECUTIVE,418,816000.0,duplicate_composite_key
654106,654106,2014-07,BUKIT MERAH,5 ROOM,84,788000.0,duplicate_composite_key
642042,642042,2013-09,QUEENSTOWN,5 ROOM,55,775000.0,duplicate_composite_key
634348,634348,2013-04,CLEMENTI,5 ROOM,413,765000.0,duplicate_composite_key
961984,961984,2015-09,QUEENSTOWN,4 ROOM,61C,765000.0,duplicate_composite_key
630307,630307,2013-01,BUKIT MERAH,4 ROOM,26D,754000.0,duplicate_composite_key


### Deduplication Findings

Deduplication quarantines 1,383 surplus records and retains 83,750 of the 85,133 inputs. This differs from the 1,595 master-profile surplus because reference validation runs first.

Total quarantine is now 8,794. Retained and quarantined records reconcile to the 92,544 in-scope inputs.

<h3 style="color: #4EA1FF;">6. Identify Potential Price Anomalies</h3>

The final screen fits thresholds to the validated, deduplicated population. It uses the same year, town, and flat-type groups and three-IQR fences as profiling.

Groups need at least 30 records and a positive IQR. Screening status distinguishes within-fence prices, high or low candidates, and unassessed records. Source references are retained explicitly across the statistics merge.

In [75]:
MIN_GROUP_ROWS = 30
IQR_MULTIPLIER = 3

price_screened = deduplicated.copy()

# The merge resets the index; retain the source reference as a column.
price_screened.insert(
    0,
    "source_record_id",
    price_screened.index,
)

price_screened["year"] = price_screened["month"].str[:4]

price_screened["price_per_sqm"] = (
    price_screened["resale_price"]
    / price_screened["floor_area_sqm"]
)

group_columns = ["year", "town", "flat_type"]

screening_stats = (
    price_screened.groupby(group_columns, observed=True)
    .agg(
        group_rows=("price_per_sqm", "count"),
        q1=("price_per_sqm", lambda values: values.quantile(0.25)),
        median=("price_per_sqm", "median"),
        q3=("price_per_sqm", lambda values: values.quantile(0.75)),
    )
    .reset_index()
)

screening_stats["iqr"] = (
    screening_stats["q3"] - screening_stats["q1"]
)

screening_stats["lower_fence"] = (
    screening_stats["q1"] - IQR_MULTIPLIER * screening_stats["iqr"]
)

screening_stats["upper_fence"] = (
    screening_stats["q3"] + IQR_MULTIPLIER * screening_stats["iqr"]
)

price_screened = price_screened.merge(
    screening_stats,
    on=group_columns,
    how="left",
    validate="many_to_one",
)

# Small or zero-IQR groups remain unassessed rather than passing implicitly.
eligible = (
    price_screened["group_rows"].ge(MIN_GROUP_ROWS)
    & price_screened["iqr"].gt(0)
    & price_screened["price_per_sqm"].notna()
)

below_fence = (
    eligible
    & price_screened["price_per_sqm"].lt(
        price_screened["lower_fence"]
    )
)

above_fence = (
    eligible
    & price_screened["price_per_sqm"].gt(
        price_screened["upper_fence"]
    )
)

price_screened["price_screening_status"] = (
    "not_assessed_insufficient_group"
)

price_screened.loc[
    price_screened["group_rows"].ge(MIN_GROUP_ROWS),
    "price_screening_status",
] = "not_assessed_zero_iqr_or_missing_metric"

price_screened.loc[
    eligible,
    "price_screening_status",
] = "within_fences"

price_screened.loc[
    below_fence,
    "price_screening_status",
] = "potential_low_price"

price_screened.loc[
    above_fence,
    "price_screening_status",
] = "potential_high_price"

price_screened["potential_price_anomaly"] = (
    below_fence | above_fence
)

if len(price_screened) != len(deduplicated):
    raise ValueError("Price screening changed the record count.")

if price_screened["source_record_id"].duplicated().any():
    raise ValueError("Price screening duplicated source records.")

display(
    price_screened["price_screening_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="rows")
)

display(
    price_screened.loc[
        price_screened["potential_price_anomaly"],
        [
            "source_record_id",
            "month",
            "town",
            "flat_type",
            "floor_area_sqm",
            "resale_price",
            "price_per_sqm",
            "group_rows",
            "lower_fence",
            "upper_fence",
            "price_screening_status",
        ],
    ].head(20)
)

,status,rows
0,within_fences,81557
1,not_assessed_insufficient_group,1884
2,potential_high_price,306
3,potential_low_price,3


,source_record_id,month,town,flat_type,floor_area_sqm,resale_price,price_per_sqm,group_rows,lower_fence,upper_fence,price_screening_status
826,607647,2012-01,KALLANG/WHAMPOA,3 ROOM,79.0,705000.0,8924.050633,244,3214.686926,7906.821175,potential_high_price
1590,608414,2012-02,ANG MO KIO,4 ROOM,90.0,642000.0,7133.333333,203,3325.426042,6924.051868,potential_high_price
1658,608484,2012-02,BEDOK,4 ROOM,85.0,598000.0,7035.294118,333,3078.714763,6536.072623,potential_high_price
3236,616926,2012-06,ANG MO KIO,4 ROOM,91.0,638000.0,7010.989011,203,3325.426042,6924.051868,potential_high_price
4204,617908,2012-06,KALLANG/WHAMPOA,3 ROOM,92.0,825000.0,8967.391304,244,3214.686926,7906.821175,potential_high_price
4410,618120,2012-06,QUEENSTOWN,3 ROOM,81.0,752800.0,9293.827160,248,3184.506725,9039.905213,potential_high_price
5531,619260,2012-07,BUKIT MERAH,3 ROOM,60.0,528000.0,8800.000000,214,3119.463869,8638.286713,potential_high_price
5859,619593,2012-07,GEYLANG,3 ROOM,60.0,450000.0,7500.000000,232,3261.964190,7341.662450,potential_high_price
5861,619595,2012-07,GEYLANG,3 ROOM,60.0,460000.0,7666.666667,232,3261.964190,7341.662450,potential_high_price
6282,620029,2012-07,KALLANG/WHAMPOA,3 ROOM,94.0,810000.0,8617.021277,244,3214.686926,7906.821175,potential_high_price


### Price Candidate Routing

The final screen finds 309 candidates among 81,866 evaluated records. Another 1,884 records have insufficient group sizes.

Candidates are quarantined for review with their thresholds and high/low reasons. Unassessed records remain in the working dataset, with status retained in the audit output. Thresholds are applied once; repeated removal and refitting would change the exclusion policy.

In [76]:
anomaly_mask = price_screened["potential_price_anomaly"]

quarantined_prices = price_screened.loc[anomaly_mask].copy()

quarantined_prices["quarantine_reasons"] = (
    quarantined_prices["price_screening_status"]
)

retained_ids = price_screened.loc[
    ~anomaly_mask,
    "source_record_id",
]

# Select source records by reference, not by the post-merge index.
cleaned_candidate = deduplicated.loc[retained_ids].copy()

quarantined = pd.concat(
    [
        quarantined_dates,
        quarantined_validation,
        quarantined_lease,
        quarantined_duplicates,
        quarantined_prices,
    ],
    ignore_index=True,
    sort=False,
)

if quarantined["source_record_id"].duplicated().any():
    raise ValueError("A source record appears more than once in quarantine.")

if cleaned_candidate.index.isin(
    quarantined["source_record_id"]
).any():
    raise ValueError("Retained and quarantined records overlap.")

if (
    len(cleaned_candidate) + len(quarantined_prices)
    != len(deduplicated)
):
    raise ValueError("Price routing counts do not reconcile.")

accounted_rows = (
    len(cleaned_candidate)
    + len(quarantined)
    + scope_exclusion_count
)

if accounted_rows != len(combined):
    raise ValueError("Pipeline outputs do not reconcile to source rows.")

print(f"Potential price anomalies quarantined: {len(quarantined_prices):,}")
print(f"Records retained: {len(cleaned_candidate):,}")
print(f"Total quarantined: {len(quarantined):,}")

Potential price anomalies quarantined: 309
Records retained: 83,441
Total quarantined: 9,103


### Final Price Findings

Price routing quarantines 306 high and three low candidates. The working dataset contains 83,441 records, including 1,884 unassessed by this heuristic.

Quarantine contains 9,103 records: 7,411 reference failures, 1,383 duplicate exclusions, and 309 price candidates. The two dispositions reconcile to all 92,544 in-scope records.

<h3 style="color: #4EA1FF;">7. Apply Additional Validation Rules</h3>

Prices and floor areas must be numeric, finite, and positive. Finiteness is checked explicitly because infinity can satisfy a positivity check.

These checks currently run after price screening. For future invalid inputs, numeric enforcement should move before arithmetic so invalid metrics cannot affect thresholds. Current profiling found no such inputs.

In [78]:
additional_numeric = cleaned_candidate[
    ["resale_price", "floor_area_sqm"]
].apply(pd.to_numeric, errors="coerce")

additional_checks = pd.DataFrame(index=cleaned_candidate.index)

for column in additional_numeric.columns:
    values = additional_numeric[column]

    # Explicit finiteness catches infinity, which can otherwise pass positivity.
    finite = pd.Series(
        np.isfinite(
            values.to_numpy(dtype="float64", na_value=np.nan)
        ),
        index=values.index,
    )

    additional_checks[f"{column}_missing_or_nonnumeric"] = (
        values.isna()
    )

    additional_checks[f"{column}_nonfinite"] = (
        values.notna() & ~finite
    )

    additional_checks[f"{column}_nonpositive"] = (
        values.notna() & values.le(0)
    )

display(
    additional_checks.sum()
    .rename_axis("rule")
    .reset_index(name="failed_rows")
)

print(
    "Records failing additional numeric checks:",
    int(additional_checks.any(axis=1).sum()),
)

,rule,failed_rows
0,resale_price_missing_or_nonnumeric,0
1,resale_price_nonfinite,0
2,resale_price_nonpositive,0
3,floor_area_sqm_missing_or_nonnumeric,0
4,floor_area_sqm_nonfinite,0
5,floor_area_sqm_nonpositive,0


Records failing additional numeric checks: 0


### Identifier and Storey Structure

Street names must be nonblank; block identifiers must contain an ASCII digit. No suffix restriction is imposed without an authoritative naming rule.

Storey ranges must use `NN TO NN` with positive, ordered bounds. This validates structure independently of reference membership. Outer whitespace is reported separately and source text is left intact.

In [79]:
street = cleaned_candidate["street_name"].astype("string")
block = cleaned_candidate["block"].astype("string")
storey = cleaned_candidate["storey_range"].astype("string")

additional_checks["street_name_missing_or_blank"] = (
    street.isna()
    | street.str.strip().eq("").fillna(False)
)

additional_checks["block_missing_or_without_digits"] = (
    ~block.str.contains(r"[0-9]", regex=True, na=False)
)

storey_bounds = storey.str.extract(
    r"^([0-9]{2}) TO ([0-9]{2})$"
).apply(pd.to_numeric, errors="coerce")

valid_storey_format = storey.str.fullmatch(
    r"[0-9]{2} TO [0-9]{2}",
    na=False,
)

additional_checks["storey_range_invalid_format"] = (
    ~valid_storey_format
)

additional_checks["storey_range_invalid_bounds"] = (
    valid_storey_format
    & (
        storey_bounds[0].le(0)
        | storey_bounds[1].le(0)
        | storey_bounds[0].gt(storey_bounds[1])
    )
)

# Whitespace is diagnostic here; no normalization or exclusion is applied.
whitespace_counts = {}

for column in text_columns:
    values = cleaned_candidate[column].astype("string")

    whitespace_counts[column] = int(
        values.ne(values.str.strip()).fillna(False).sum()
    )

display(
    additional_checks.sum()
    .rename_axis("rule")
    .reset_index(name="failed_rows")
)

display(
    pd.Series(whitespace_counts)
    .rename_axis("column")
    .reset_index(name="rows_with_outer_whitespace")
)

print(
    "Records failing any additional rule:",
    int(additional_checks.any(axis=1).sum()),
)

,rule,failed_rows
0,resale_price_missing_or_nonnumeric,0
1,resale_price_nonfinite,0
2,resale_price_nonpositive,0
3,floor_area_sqm_missing_or_nonnumeric,0
4,floor_area_sqm_nonfinite,0
5,floor_area_sqm_nonpositive,0
6,street_name_missing_or_blank,0
7,block_missing_or_without_digits,0
8,storey_range_invalid_format,0
9,storey_range_invalid_bounds,0


,column,rows_with_outer_whitespace
0,month,0
1,town,0
2,flat_type,0
3,block,0
4,street_name,0
5,storey_range,0
6,flat_model,0
7,remaining_lease,0


Records failing any additional rule: 0


### Additional Rule Findings

All ten additional numeric and structural checks pass. The inspected text fields have no outer whitespace. No further records require exclusion under these rules.

### Final Cleaning Reconciliation

Additional failures retain their reasons and source references. Quarantine is rebuilt from the stage outputs on each run, preventing duplicate appends.

The remaining records form `cleaned`. Count reconciliation and reference checks confirm no overlap with quarantine.

In [81]:
failed_additional = additional_checks.any(axis=1)

additional_failure_reasons = additional_checks.apply(
    lambda row: "; ".join(row.index[row]),
    axis=1,
)

quarantined_additional = cleaned_candidate.loc[
    failed_additional
].copy()

quarantined_additional.insert(
    0,
    "source_record_id",
    quarantined_additional.index,
)

quarantined_additional["quarantine_reasons"] = (
    additional_failure_reasons.loc[failed_additional]
)

cleaned = cleaned_candidate.loc[~failed_additional].copy()

# Rebuild the complete quarantine once, retaining each exclusion stage.
quarantined = pd.concat(
    [
        quarantined_dates,
        quarantined_validation,
        quarantined_lease,
        quarantined_duplicates,
        quarantined_prices,
        quarantined_additional,
    ],
    ignore_index=True,
    sort=False,
)

if quarantined["source_record_id"].duplicated().any():
    raise ValueError("A source record appears more than once in quarantine.")

if cleaned.index.isin(quarantined["source_record_id"]).any():
    raise ValueError("Cleaned and quarantined records overlap.")

if (
    len(cleaned) + len(quarantined_additional)
    != len(cleaned_candidate)
):
    raise ValueError("Additional validation counts do not reconcile.")

accounted_rows = (
    len(cleaned)
    + len(quarantined)
    + scope_exclusion_count
)

if accounted_rows != len(combined):
    raise ValueError("Final cleaning outputs do not reconcile.")

print(f"Additional failures quarantined: {len(quarantined_additional):,}")
print(f"Cleaned records: {len(cleaned):,}")
print(f"Total quarantined: {len(quarantined):,}")
print(f"Outside-period exclusions: {scope_exclusion_count:,}")

Additional failures quarantined: 0
Cleaned records: 83,441
Total quarantined: 9,103
Outside-period exclusions: 894,004


<h3 style="color: #4EA1FF;">8. Document Insights and Assumptions</h3>

[Insights and assumptions](docs/insights_and_assumptions.md) records the source findings, date-reference interpretation, lease-start assumption, duplicate selection policy, anomaly thresholds, identifier collisions, and final reconciliation.

Counts refer to the current source snapshot. The document also records limitations and remaining submission checks.

## Data Transformation

<h3 style="color: #4EA1FF;">9. Construct the Resale Identifier</h3>

Transformation starts from `cleaned` and confirms original-key uniqueness. Group average prices are calculated by transaction month, town, and flat type; `identifier_audit` retains the averages used in this run.

The identifier bullets printed under Requirement 10 are interpreted as Requirement 9's construction rules, followed by hashing. The prescribed format can repeat across distinct keys, so collisions are measured separately.

In [83]:
if cleaned.duplicated(subset=key_columns).any():
    raise ValueError("Repeated composite keys remain in the cleaned data.")

identifier_audit = cleaned[
    ["month", "town", "flat_type", "block"]
].copy()

# The identifier average uses the final cleaned population.
identifier_audit["group_average_price"] = (
    cleaned.groupby(
        ["month", "town", "flat_type"],
        observed=True,
    )["resale_price"]
    .transform("mean")
)

display(identifier_audit.head(10))

,month,town,flat_type,block,group_average_price
606808,2012-01,ANG MO KIO,2 ROOM,406,256966.666667
606809,2012-01,ANG MO KIO,2 ROOM,314,256966.666667
606810,2012-01,ANG MO KIO,2 ROOM,314,256966.666667
606811,2012-01,ANG MO KIO,2 ROOM,170,256966.666667
606812,2012-01,ANG MO KIO,2 ROOM,174,256966.666667
606813,2012-01,ANG MO KIO,2 ROOM,508,256966.666667
606814,2012-01,ANG MO KIO,3 ROOM,174,348328.654737
606815,2012-01,ANG MO KIO,3 ROOM,216,348328.654737
606816,2012-01,ANG MO KIO,3 ROOM,332,348328.654737
606817,2012-01,ANG MO KIO,3 ROOM,418,348328.654737


### Identifier Assembly

The readable identifier combines `S`, the first three block digits padded on the left, the first two digits of the integer group average, the transaction month, and the town initial.

The average is truncated rather than rounded up. Construction stops if required digits are unavailable. The specified format is preserved even when identifiers repeat.

In [84]:
block_digits = (
    identifier_audit["block"]
    .str.replace(r"[^0-9]", "", regex=True)
)

if block_digits.eq("").any():
    raise ValueError("A block has no digits for identifier construction.")

block_component = block_digits.str[:3].str.zfill(3)

# Truncate before taking leading digits; rounding could change the prefix.
average_integer = (
    identifier_audit["group_average_price"]
    .floordiv(1)
    .astype("int64")
    .astype("string")
)

if average_integer.str.len().lt(2).any():
    raise ValueError("A group average has fewer than two digits.")

price_component = average_integer.str[:2]
month_component = identifier_audit["month"].str[5:7]
town_component = identifier_audit["town"].str[:1]

transformed = cleaned.copy()

transformed["Resale Identifier"] = (
    "S"
    + block_component
    + price_component
    + month_component
    + town_component
)

valid_identifier = transformed["Resale Identifier"].str.fullmatch(
    r"S[0-9]{7}[A-Z]",
    na=False,
)

if not valid_identifier.all():
    raise ValueError("An identifier does not match the prescribed structure.")

# Readable identifiers may repeat even though original keys are unique.
identifier_counts = transformed["Resale Identifier"].value_counts()
repeated_identifiers = identifier_counts.loc[identifier_counts.gt(1)]

print(f"Transformed records: {len(transformed):,}")
print(f"Distinct identifiers: {len(identifier_counts):,}")
print(f"Identifiers shared by multiple records: {len(repeated_identifiers):,}")
print(f"Records sharing identifiers: {repeated_identifiers.sum():,}")

display(
    transformed[
        [
            "month",
            "town",
            "flat_type",
            "block",
            "resale_price",
            "Resale Identifier",
        ]
    ].head(10)
)

Transformed records: 83,441
Distinct identifiers: 71,591
Identifiers shared by multiple records: 9,635
Records sharing identifiers: 21,485


,month,town,flat_type,block,resale_price,Resale Identifier
606808,2012-01,ANG MO KIO,2 ROOM,406,257800.0,S4062501A
606809,2012-01,ANG MO KIO,2 ROOM,314,263000.0,S3142501A
606810,2012-01,ANG MO KIO,2 ROOM,314,275000.0,S3142501A
606811,2012-01,ANG MO KIO,2 ROOM,170,260000.0,S1702501A
606812,2012-01,ANG MO KIO,2 ROOM,174,226000.0,S1742501A
606813,2012-01,ANG MO KIO,2 ROOM,508,260000.0,S5082501A
606814,2012-01,ANG MO KIO,3 ROOM,174,281000.0,S1743401A
606815,2012-01,ANG MO KIO,3 ROOM,216,375000.0,S2163401A
606816,2012-01,ANG MO KIO,3 ROOM,332,350000.0,S3323401A
606817,2012-01,ANG MO KIO,3 ROOM,418,420000.0,S4183401A


### Identifier Findings

All 83,441 records receive a readable identifier. There are 71,591 distinct values; 9,635 values are shared by 21,485 records.

The format omits year and several key attributes, so repeated identifiers are expected. Hashing the readable value alone would preserve those repetitions. The hash input therefore includes the original key, as documented below.

<h3 style="color: #4EA1FF;">10. Hash the Identifier and Verify Uniqueness</h3>

SHA-256 hashes the readable identifier together with the original composite key. This extends the hash input to resolve the conflict between the prescribed repeating format and the requested uniqueness, while leaving the readable identifier intact.

JSON serialization fixes key order, includes column names, normalizes nulls, and rejects nonfinite numbers. Price, derived fields, and source row references are excluded from the key. Digest format and uniqueness are checked before export.

Hashes are deterministic for unchanged inputs and serialization. They are not guaranteed persistent identifiers across source revisions or a guarantee of anonymization.

In [86]:
# Fixed ordering keeps serialization independent of DataFrame column order.
hash_key_columns = sorted(key_columns)


def hash_resale_record(row):
    composite_key = []

    for column in hash_key_columns:
        value = row[column]

        if pd.isna(value):
            normalized = None
        elif isinstance(value, np.generic):
            normalized = value.item()
        else:
            normalized = value

        composite_key.append([column, normalized])

    # Include the original key to distinguish repeating readable identifiers.
    payload = {
        "version": 1,
        "resale_identifier": row["Resale Identifier"],
        "composite_key": composite_key,
    }

    # Structured JSON preserves field boundaries and rejects NaN or infinity.
    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        separators=(",", ":"),
        allow_nan=False,
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


hashed = transformed.copy()

hashed["Hashed Resale Identifier"] = transformed.apply(
    hash_resale_record,
    axis=1,
)

valid_hash_format = hashed["Hashed Resale Identifier"].str.fullmatch(
    r"[0-9a-f]{64}",
    na=False,
)

if not valid_hash_format.all():
    raise ValueError("An invalid SHA-256 digest was generated.")

if hashed["Hashed Resale Identifier"].duplicated().any():
    raise ValueError("Repeated hashes found; investigate before exporting.")

if len(hashed) != len(cleaned):
    raise ValueError("Hashing changed the cleaned record count.")

print(f"Hashed records: {len(hashed):,}")
print(
    "Distinct hashes:",
    f"{hashed['Hashed Resale Identifier'].nunique():,}",
)

display(
    hashed[
        [
            "month",
            "town",
            "block",
            "Resale Identifier",
            "Hashed Resale Identifier",
        ]
    ].head(10)
)

Hashed records: 83,441
Distinct hashes: 83,441


,month,town,block,Resale Identifier,Hashed Resale Identifier
606808,2012-01,ANG MO KIO,406,S4062501A,9a43c7d6841998c8a973d328dc7f050155236d19b2b485...
606809,2012-01,ANG MO KIO,314,S3142501A,3bd083993f74e82f575c5f8b099f0a33a8c85d0342b111...
606810,2012-01,ANG MO KIO,314,S3142501A,2602f4b2f59e9ceac7b627ef44203e17a1e6b36d364d29...
606811,2012-01,ANG MO KIO,170,S1702501A,38ef554f87c9a8bdb69b612523db1e57af59637f4de2d6...
606812,2012-01,ANG MO KIO,174,S1742501A,df94b6dce31a1cffc9e56f324fbfbbc627851277372606...
606813,2012-01,ANG MO KIO,508,S5082501A,b2869a33a5337b98c75c5b0789626184a4f8826ed15b07...
606814,2012-01,ANG MO KIO,174,S1743401A,ef5a333c06f12a11dd1c9672acd57469ed921497105d1f...
606815,2012-01,ANG MO KIO,216,S2163401A,15480754cd3d7c1bdbb43429d928e92089d8793a02e317...
606816,2012-01,ANG MO KIO,332,S3323401A,dd5e315fd9cd893212858f80a1d25d3487bac8a9ac6d17...
606817,2012-01,ANG MO KIO,418,S4183401A,616ba35a0668f1c693cdfe012b267862b2fb2e8d870572...


## Data Output Requirements

<h3 style="color: #4EA1FF;">11–12. Export the Five Required Output Groups</h3>

Raw files remain unchanged in `data/raw/`. Cleaned, Transformed, Quarantined, and Hashed CSVs are written separately under `data/output/`. The Hashed output retains the readable identifier alongside its digest.

The separate price audit includes thresholds and statuses for all records entering final screening. Source references are exported explicitly because CSV does not preserve the notebook index.

Counts and quarantine separation are checked before writing. Each completed temporary file replaces its destination; replacement applies per file, without making the whole export one transaction.

In [87]:
OUTPUT_DIR = PART1_DIR / "data" / "output"

output_groups = {
    "cleaned": cleaned,
    "transformed": transformed,
    "quarantined": quarantined,
    "hashed": hashed,
}

if not (len(cleaned) == len(transformed) == len(hashed)):
    raise ValueError("Accepted output counts differ.")

if quarantined["source_record_id"].duplicated().any():
    raise ValueError("Repeated source references found in quarantine.")

if cleaned.index.isin(quarantined["source_record_id"]).any():
    raise ValueError("Cleaned and quarantined outputs overlap.")

if (
    len(cleaned) + len(quarantined) + scope_exclusion_count
    != len(combined)
):
    raise ValueError("Output counts do not reconcile to source rows.")

export_summary = []

for group, frame in output_groups.items():
    group_dir = OUTPUT_DIR / group
    group_dir.mkdir(parents=True, exist_ok=True)

    export_frame = frame.copy()

    # CSV omits the index, so export the run-specific reference explicitly.
    if "source_record_id" not in export_frame.columns:
        export_frame.insert(
            0,
            "source_record_id",
            export_frame.index,
        )

    destination = group_dir / f"{group}.csv"
    temporary = destination.with_suffix(".csv.part")

    export_frame.to_csv(
        temporary,
        index=False,
        encoding="utf-8",
    )

    # Replace this file only after its write completes.
    temporary.replace(destination)

    export_summary.append({
        "group": group,
        "rows": len(export_frame),
        "file": str(destination.relative_to(PART1_DIR)),
    })

AUDIT_DIR = OUTPUT_DIR / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

audit_path = AUDIT_DIR / "price_screening.csv"
audit_temporary = audit_path.with_suffix(".csv.part")

price_screened.to_csv(
    audit_temporary,
    index=False,
    encoding="utf-8",
)

audit_temporary.replace(audit_path)

display(pd.DataFrame(export_summary))

print(f"Raw output: {RAW_DIR}")
print(f"Price-screening audit: {audit_path}")

,group,rows,file
0,cleaned,83441,data\output\cleaned\cleaned.csv
1,transformed,83441,data\output\transformed\transformed.csv
2,quarantined,9103,data\output\quarantined\quarantined.csv
3,hashed,83441,data\output\hashed\hashed.csv


Raw output: c:\Users\mirza\Desktop\Personal Projects\hdb-data-engineering-technical-test\part-1\data\raw
Price-screening audit: c:\Users\mirza\Desktop\Personal Projects\hdb-data-engineering-technical-test\part-1\data\output\audit\price_screening.csv
